# PT Create HTMLs — Fast
Run as often as needed (odds update, tips update, etc.).
Requires `PT_Vorarbeiten.ipynb` to have been run once today.

## 1. Imports

In [10]:
import os, re, json, pickle, warnings
import numpy as np
import pandas as pd
import unicodedata
import requests
from datetime import datetime, date
from scipy.stats import ttest_ind
from tqdm.notebook import tqdm
from google.colab import userdata
import pytz

In [ ]:
# papermill parameters — injected by PT_WORKFLOW.ipynb when run automatically
GITHUB_TOKEN = None


## 2. Mount Drive

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Config & Load Pre-computed Data

In [12]:
BASE_HIST = "/content/drive/MyDrive/PT"       if os.path.exists("/content/drive/MyDrive/PT")       else "."
BASE_OUT  = "/content/drive/MyDrive/PT/races" if os.path.exists("/content/drive/MyDrive/PT/races") else "."
TODAY     = date.today().strftime("%Y-%m-%d")

# ── Pre-computed by Vorarbeiten ───────────────────────────────────────────────
runners         = pd.read_parquet(f"{BASE_HIST}/runners_processed.parquet")
df_with_ratings = pd.read_parquet(f"{BASE_HIST}/df_with_ratings.parquet")

with open(f"{BASE_HIST}/notepad_flags_{TODAY}.pkl", "rb") as f:
    notepad_flags = pickle.load(f)

with open(f"{BASE_HIST}/precomputed_tdy_{TODAY}.pkl", "rb") as f:
    _tdy = pickle.load(f)
runners_tdy = _tdy['runners_tdy']
webTips_tdy = _tdy['webTips_tdy']
odds_tdy    = _tdy['odds_tdy']
races_tdy   = _tdy['races_tdy']
today_tips  = _tdy['today_tips']

print(f"✅ Loaded runners_processed  ({len(runners):,} rows)")
print(f"✅ Loaded df_with_ratings    ({len(df_with_ratings):,} rows)")
print(f"✅ Loaded notepad_flags      ({len(notepad_flags)} flags)")
print(f"✅ Loaded precomputed_tdy    ({len(runners_tdy)} runners, {runners_tdy['raceId'].nunique()} races)")


✅ Loaded runners_processed  (302,714 rows)
✅ Loaded df_with_ratings    (259,971 rows)
✅ Loaded notepad_flags      (76 flags)


## 5. Fetch PMU Odds  ← always fetches fresh

In [15]:
GITHUB_TOKEN = GITHUB_TOKEN or userdata.get("GITHUB_TOKEN")
REPO         = "arauch6363-crypto/pmu-tracker"

def normalize_name(name: str) -> str:
    name = unicodedata.normalize("NFD", str(name).strip())
    name = "".join(c for c in name if unicodedata.category(c) != "Mn")
    return re.sub(r"^#\d+\s+", "", name).upper().strip()

def filter_before_race(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=False)
    race_start = pd.to_datetime(df["timestamp"].dt.date.astype(str) + " " + df["heure"])
    return df[df["timestamp"] < race_start]

url  = f"https://raw.githubusercontent.com/{REPO}/main/history/odds_{TODAY}.json"
resp = requests.get(url, headers={"Authorization": f"token {GITHUB_TOKEN}"})
resp.raise_for_status()
data = resp.json()

rows = [
    {"timestamp": ts, "race": rk, "hippodrome": race["hippodrome"],
     "heure": race["heure"], "horse": horse, "odds": d["odds"],
     "tendance": d["tendance"], "magnitude": d["magnitude"], "favoris": d["favoris"]}
    for ts, snapshot in data.items()
    for rk, race in snapshot.items()
    for horse, d in race["horses"].items()
]
df_odds = pd.DataFrame(rows).sort_values(["race","heure","horse","timestamp"]).reset_index(drop=True)
pmu_odds_history = filter_before_race(df_odds)
pmu_odds_history["horse"] = pmu_odds_history["horse"].apply(normalize_name)
runners_tdy["_key"] = runners_tdy["horseName"].apply(normalize_name)
pmu_odds_history = (
    pmu_odds_history
    .merge(runners_tdy[["_key","horseName"]], left_on="horse", right_on="_key", how="left")
    .drop(columns=["_key","horse"])
    .dropna(subset=["horseName"])
    .reset_index(drop=True)
)
print(f"✅ PMU odds fetched  ({len(pmu_odds_history):,} rows)")

✅ PMU odds fetched  (3,481 rows)


## 6. Load HTML Functions

In [16]:
import sys, os

_module_path = os.path.join(BASE_HIST, 'pt_html_functions.py')
print(f'Looking for module at: {_module_path}')
print(f'File exists: {os.path.exists(_module_path)}')

if not os.path.exists(_module_path):
    raise FileNotFoundError(f'pt_html_functions.py not found at {_module_path}')

# %run executes the file directly — no sys.path / module cache issues
%run {_module_path}
print(f'✅ pt_html_functions loaded from {_module_path}')

Looking for module at: /content/drive/MyDrive/PT/pt_html_functions.py
File exists: True
✅ pt_html_functions loaded from /content/drive/MyDrive/PT/pt_html_functions.py


## 7. Export HTMLs  ← the fast part

In [ ]:
os.makedirs(BASE_OUT, exist_ok=True)

update_all_races_html_odds(
    output_dir       = BASE_OUT,
    today_date       = TODAY,
    pmu_odds_history = pmu_odds_history,
)

🏇 Exporting 7 races across 2 meetings...

📍 Chantilly
  [██░░░░░░░░░░░░░░░░░░] 1/7  Prix du Champ d'Alouette
  [█████░░░░░░░░░░░░░░░] 2/7  Prix Esso
  [████████░░░░░░░░░░░░] 3/7  Prix Otto
  [███████████░░░░░░░░░] 4/7  Prix de la Biche

📍 La Teste
  [██████████████░░░░░░] 5/7  Prix des Testerines
  [█████████████████░░░] 6/7  Prix de la Société Hippique de Gabarret
  [████████████████████] 7/7  Prix de l'APPE

✅ Done — 7 files saved to /content/drive/MyDrive/PT/races


['2026-04-22__Chantilly__Prix_du_Champ_d_Alouette.html',
 '2026-04-22__Chantilly__Prix_Esso.html',
 '2026-04-22__Chantilly__Prix_Otto.html',
 '2026-04-22__Chantilly__Prix_de_la_Biche.html',
 '2026-04-22__La_Teste__Prix_des_Testerines.html',
 '2026-04-22__La_Teste__Prix_de_la_Soci_t__Hippique_de_Gabarret.html',
 '2026-04-22__La_Teste__Prix_de_l_APPE.html']